# Lekcja 01 - Wprowadzenie do Agentów AI

Witamy na pierwszej lekcji kursu **Agenci AI dla początkujących**!

**Agent AI** to program, który używa dużego modelu językowego (LLM) jako swojego silnika rozumowania i potrafi podejmować *działania* w świecie rzeczywistym — wywołując API, zapytując bazy danych lub uruchamiając kod — aby osiągnąć cel w imieniu użytkownika.

W tym notatniku zbudujesz swojego pierwszego agenta: **Agenta Podróży**, który poleca miejsca na wakacje. Po drodze nauczysz się jak:

1. Połączyć się z Microsoft Foundry Agent Service używając **Microsoft Agent Framework**.
2. Dać agentowi **narzędzie** — zwykłą funkcję Pythona, którą może wywołać.
3. Uruchomić agenta i sprawdzić jego odpowiedź.
4. Przesyłać odpowiedź agenta token po tokenie.


## Konfiguracja

Przed uruchomieniem tego notatnika upewnij się, że masz:

1. **Projekt Microsoft Foundry** z wdrożonym modelem czatu (np. `gpt-4.1-mini`).
2. **Zalogowano się za pomocą Azure CLI** — uruchom w terminalu polecenie `az login`.
3. **Ustawione wymagane zmienne środowiskowe:**
   - `AZURE_AI_PROJECT_ENDPOINT` — punkt końcowy Twojego projektu Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nazwa wdrożonego modelu.

Poniższa komórka instaluje wymagane pakiety Pythona.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tworzenie Twojego Pierwszego Agenta

Agent potrzebuje dwóch rzeczy:

- **Instrukcji**, które mówią mu *kim jest* i *jak się zachowywać* (systemowy prompt).
- **Narzędzi** — funkcji Pythona oznaczonych dekoratorem `@tool`, które agent może wywołać, aby uzyskać informacje lub wykonać działania.

Poniżej definiujemy proste narzędzie, które zwraca listę popularnych miejsc wakacyjnych. Agent użyje tego narzędzia, gdy użytkownik poprosi o rekomendacje podróżnicze.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Odpowiedzi strumieniowe

Aby uzyskać bardziej interaktywne doświadczenie, możesz **strumieniować** odpowiedź agenta. Zamiast czekać na pełną odpowiedź, agent przekazuje fragmenty tekstu w miarę ich generowania. Jest to szczególnie przydatne w interfejsach czatu, gdzie chcesz wyświetlać wyniki w czasie rzeczywistym.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się jak:

- **Utworzyć providera**, który łączy się z Microsoft Foundry Agent Service za pomocą `FoundryChatClient`.
- **Zdefiniować narzędzie** używając dekoratora `@tool`, aby agent mógł wywoływać Twoje funkcje Pythona.
- **Uruchomić agenta** z wiadomością od użytkownika i wyświetlić jego odpowiedź.
- **Przesyłać odpowiedzi strumieniowo** dla wyjścia w czasie rzeczywistym.

W następnej lekcji bardziej szczegółowo przyjrzymy się agentycznym frameworkom i nauczymy się, jak dawać agentom potężniejsze narzędzia oraz zdolności do wieloetapowego rozumowania.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
